# Setup environment

In [1]:
%reload_ext autoreload
%autoreload 2

**Introduction**

In the real world of machine learning, particularly in fraud detection, disease diagnosis, or anomaly detection, we often face a common challenge: imbalanced datasets. When one class significantly outnumbers the other, our models can become biased, leading to suboptimal performance where the minority class - often the one we're most interested in - gets overlooked.

## Import libraries

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
# import model libraries
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_curve, roc_curve

import pickle


#### Define support functions

## import the dataset and the model

In [ ]:
score_balanced_model = pickle.load(open('models/score_balanced_model.pkl', 'rb'))

In [ ]:
oot_X = pickle.load(open('models/oot_X.pkl', 'rb'))
oot_y = pickle.load(open('models/oot_y.pkl', 'rb'))


In [8]:
metadata_columns = ['trans_date_trans_time','gender','street','trans_num']

#model dump
import pickle
with open('models/score_balanced_model.pkl', 'wb') as f:
    pickle.dump(score_balanced_model, f)

## Run a basic model with downsampling on dataset

In [ ]:
mp = ModelPipeline(data, metadata_columns)
score_balanced_model, train_X, train_y, holdout_X, holdout_y, oot_X, oot_y = mp.main()

In [10]:
#model dump
import pickle
with open('models/score_balanced_model.pkl', 'wb') as f:
    pickle.dump(score_balanced_model, f)

In [11]:
#dump datasets
with open('models/train_X.pkl', 'wb') as f:
    pickle.dump(train_X, f)
with open('models/train_y.pkl', 'wb') as f:
    pickle.dump(train_y, f)
with open('models/holdout_X.pkl', 'wb') as f:
    pickle.dump(holdout_X, f)
with open('models/holdout_y.pkl', 'wb') as f:
    pickle.dump(holdout_y, f)
with open('models/oot_X.pkl', 'wb') as f:
    pickle.dump(oot_X, f)
with open('models/oot_y.pkl', 'wb') as f:
    pickle.dump(oot_y, f)

## Let's look at the metrics

confusion matrix metrics:
- precision
- recall
- accuracy
- F1

PR-AUC
ROC-AUC

In [12]:
report_class = ModelPerformanceReport(train_X,train_y,holdout_X,holdout_y,oot_X,oot_y)
eval_plots = EvalPlots()

Lets score the datasets

In [13]:
y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_true, y_oot_pred = report_class.predictions(score_balanced_model)

## METRICS

### CONFUSION MATRIX

<img src="/images/metrics/enhanced_confusion_matrix.png">

__metrics__
* Accuracy: 
    - Answers to: How often is the classifier correct?
    - Ratio of correct predictions to total predictions: $Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$
    
* Precision: 
    - Answers: When it predicts positive, how often is it right? 
    - Ratio of correct positive predictions to total positive predictions: $Precision = \frac{TP}{TP + FP}$

* Recall: 
    - Answers: When it's actually positive, how often does it predict it?
    - Ratio of correct positive predictions to total actual positives: $Recall = \frac{TP}{TP + FN}$

* F1 Score: 
    - Answers: 
    - Harmonic mean of precision and recall: $F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$


Lets look at an example. High accuracy might seems good, but with such unbalanced dataset high accuracy is easily obtained by scoring as 0 most of the population

In [ ]:
# metrics for the different datasets
report_class.produce_report(score_balanced_model)

In [ ]:
#Lets look at the distribution of the predictions
df = pd.DataFrame(list(map(list, zip(*[ pd.Series(y_oot_pred).value_counts( normalize=True).tolist()
                , pd.Series(y_oot_true).value_counts( normalize=True).tolist(),
                pd.Series(y_holdout_pred).value_counts( normalize=True).tolist()
                , pd.Series(y_holdout_true).value_counts( normalize=True).tolist(),
                pd.Series(y_train_pred).value_counts( normalize=True).tolist()
                ,  pd.Series(y_train_true).value_counts( normalize=True).tolist()
                ]))),
              index=pd.Index(['Non Fraud (Negative)', 'Fraud (Positive)'], name='Labels'),
              columns=pd.MultiIndex.from_product([['OOT', 'Holdout', 'Train'],['Pred', 'True']], names=['Dataset:', 'y'])
              )
df

In [ ]:
df

### PR-AUC vs ROC-AUC

**General Notions**
The PR-AUC plots provide a more realistic picture of model performance in fraud detection scenarios, as they better capture the challenges of detecting rare fraudulent transactions while maintaining reasonable precision. This is particularly important in fraud detection where both false positives (blocking legitimate transactions) and false negatives (missing fraud) have significant business implications.
In details:

ROC AUC plots can be misleading in highly imbalanced datasets because:
* They show the trade-off between True Positive Rate (TPR) and False Positive Rate (FPR): a model can achieve high ROC AUC by simply predicting the majority class
* The ROC curve might look good even when the model is not performing well on the minority class
Therefore, ROC plot is less sensitive to class imvalance and might lead to optimistic considerations. In addition, it gives no insight on the capability of the model to detect rare event.

PR AUC plots are more informative because:
* They show the trade-off between Precision and Recall, therefore focusing on the positive class (fraudulent transactions).
* They better reflect the model's ability to handle the minority class, providing a clearer picture of the model's practical utility


__Commenting our Plots__

Looking at the model performance across different datasets, the PR-AUC shows that the model is overfitting, while the ROC-AUC gives a stable very high performance (around 0.99).
The significant 0.4 drop of PR from Training to Out-of-Time (OOT) indicates poor generalization to new data and a struggle to maintain precision while keeping recall high. All indicators of overfitting.


**Best Practices for Fraud Detection:**

* Use PR-AUC as the primary evaluation metric
* Monitor both precision and recall trade-offs
* Consider the business impact of false positives vs false negatives
* Implement proper sampling techniques within cross-validation folds
* Use stratified cross-validation to maintain class distribution




In [ ]:
report_class.produce_pr_auc_report(score_balanced_model)

In [ ]:
report_class.plot_eval_roc_auc_report(score_balanced_model)

In [ ]:
y_train_pred, y_train_true, y_holdout_pred, y_holdout_true, y_oot_pred, y_oot_true  =   report_class.proba_predictions(score_balanced_model)
report_class.plot_eval_tpr_fpr_curve(y_train_true, y_train_pred, y_holdout_true, y_holdout_pred, y_oot_true, y_oot_pred)

In [ ]:
report_class.produce_proba_report(score_balanced_model)

### Lets see the effects of a lower fraud rate (fraud population varies)

In [ ]:
from fraud_rate_perturbation_simulation import create_altered_datasets, plot_curves, plot_prediction_distribution

datasets = create_altered_datasets(oot_X[train_X.columns], oot_y, fraud_rates=[0.001, 0.005, 0.01, 0.02, 0.05])
plot_curves(datasets, score_balanced_model)

### Lets now compare PR and ROC for population drifting datasets

This analysis helps us understand:
* How well the model performs when fraud patterns change
* Whether the model is more sensitive to certain types of shifts
* The model's robustness to different types of fraud pattern changes

Following, we generate 4 shifts of the fraudulent featue distribution and then test PR and ROC curves on each new dataset.

Details on shifts:

1. Original (No Shift) - shift_factor = 0
    * This is the baseline dataset with no modifications
    * Used as a reference point to compare against other shifts
2. Positive Shift - shift_factor = 1
    * All numeric features for fraud cases are shifted upward
    * For each numeric column, fraud cases are increased by 1 standard deviation
    * This simulates fraud cases becoming more extreme in the positive direction
    * Example: If a feature has values [1, 2, 3] for fraud cases, they become [2, 3, 4]

3. Negative Shift - shift_factor = -1
    * All numeric features for fraud cases are shifted downward
    * For each numeric column, fraud cases are decreased by 1 standard deviation
    * This simulates fraud cases becoming more extreme in the negative direction
    * Example: If a feature has values [1, 2, 3] for fraud case

4. Mixed Shift - shift_factor = 0.5
    * Features for fraud cases are randomly shifted in both directions
    * For each numeric column:
        - Randomly decides whether to shift up or down
        - Uses np.random.choice([-1, 1]) to determine direction
        - Applies a 0.5 standard deviation shift in the chosen direction
    * This simulates fraud cases becoming more extreme but in different directions for different features
    * Example: If we have two features:
        - Feature 1: [1, 2, 3] might become [1.5, 2.5, 3.5] (shifted up)
        - Feature 2: [1, 2, 3] might become [0.5, 1.5, 2.5] (shifted down)

Important notes about all shifts:
* Only fraud cases (where y == 1) are modified
* Non-fraud cases remain unchanged
* After shifting, all numeric features are standardized using StandardScaler
* The fraud rate remains the same across all shifts
* The shifts help us understand how robust the model is to changes in the fraud population's characteristics

In [ ]:
from fraud_population_shift import create_shifted_datasets, plot_curves, plot_prediction_distribution

datasets = create_shifted_datasets(oot_X[train_X.columns], oot_y)
plot_curves(datasets, score_balanced_model)

In [47]:
#plot_prediction_distribution(datasets, score_balanced_model)